In [1]:
import sys
sys.path.append(r'C:\Users\liuyujie714\Desktop\TprParser\x64\Release')

import subprocess, shutil, copy
import TprParser
import numpy as np

class TprReader:
    """ @brief A wrapper of TprParser
        1. get atmic coordinates of tpr
        2. modify simulation nsteps, delta t or coordinates and save as new.tpr
    """
    def __init__(self, fname, bGRO = False, bMol2 = False, bCharge = False) -> None:
        # get internal object
        self.tprCapsule = TprParser.load(fname, bGRO, bMol2, bCharge)
    
    def set_nsteps(self, nsteps):
        """ @brief set up nsteps of tpr, same as mdp

        Parameters
        ----------
        nsteps: the nsteps of simulation

        Returns
        -------
        return True if succeed
        """
        return TprParser.set_nsteps(self.tprCapsule, nsteps)

    def set_dt(self, dt):
        """ @brief set up dt of tpr in ps, same as mdp

        Parameters
        ----------
        dt: the dt of simulation, ps

        Returns
        -------
        return True if succeed
        """
        return TprParser.set_dt(self.tprCapsule, dt)
    
    def set_coords(self, coords):
        """ @brief set up atomic coordinates of tpr

        Parameters
        ----------
        coords: a list of atom coordinates, the length must be natoms * 3

        Returns
        -------
        return True if succeed
        """
        return TprParser.set_coordinates(self.tprCapsule, coords)
    
    def set_pressure(self, epc, epct, tau_p, ref_p, compress):
        """ @brief set up pressure coulping parts of tpr

        Parameters
        ----------
        epc: pressure coupling method, No, Berendsen, ParrinelloRahman, CRescale
        epct: pressure coupling type, Isotropic, SemiIsotropic
        tau_p: the pressure coupling constant
        ref_p: a list of pressure in bar, the length must be 9
        compress: a list of compressibility in bar^-1, the length must be 9

        Returns
        -------
        return True if succeed
        """
        return TprParser.set_pressure(self.tprCapsule, epc, epct, tau_p, ref_p, compress)

    def get_coords(self):
        """ @brief get atomic coordinates from tpr

        Returns
        -------
        return a list of atom coordinates, the length is natoms * 3
        """
        return TprParser.get_coordinates(self.tprCapsule)


In [2]:
def Pressure(fname):
    reader = TprReader(fname)
    ref_p = [
        100, 0, 0,
        0, 100, 0,
        0, 0, 100
    ]
    compress = [
        4.5E-5, 0, 0,
        0, 4.5E-5, 0,
        0, 0, 4.5E-5
    ]
    assert len(ref_p) == 9
    assert len(compress) == 9
    reader.set_pressure('No', 'Isotropic', 1.0, ref_p, compress)

if __name__ == '__main__':
    Pressure("test/semiP.tpr")

: 

In [5]:
def run_cmd(cmd:str):
    ret = subprocess.run(cmd, shell=True)
    if ret.returncode != 0:
        raise Exception('\nError occurred from command: \n\t%s!!!' %cmd)
    
def MD(inittpr:str, nsteps:int = 10):
    reader = TprReader(inittpr)
    x = reader.get_coords() # get coords from tpr
    natmA = 120
    natmB = 132
    natm = natmA+natmB

    coords = np.array(x).reshape(-1, 3) # to N*3 shape
    assert natm == coords.shape[0]
    for i in range(nsteps):
        # move two molecules distance of Z axis each 2.0 A
        tempcoords = copy.deepcopy(coords)
        tempcoords[:natmA,     2] += 0.05 * i
        tempcoords[natmA:natm, 2] -= 0.05 * i
        reader.set_coords(np.array(tempcoords.flatten(), dtype=np.float32))
        # reader.set_coords(list(tempcoords.flatten()))
        # rename new.tpr to em_{i}.tpr
        suffix = inittpr.split(".tpr")[0]+"_"+str(i)
        shutil.move("new.tpr", f"{suffix}.tpr")
        run_cmd(f'gmx mdrun -deffnm {suffix} -v')
    print("Finished!")

if __name__ == '__main__':
    MD("em.tpr", 2)


[2.4709999561309814, 2.5209999084472656, 5.438000202178955, 2.4739999771118164, 2.61299991607666, 5.478000164031982, 2.4679999351501465, 2.5280001163482666, 5.339000225067139, 2.390000104904175, 2.4730000495910645, 5.4710001945495605, 2.5910000801086426, 2.447000026702881, 5.478000164031982, 2.5899999141693115, 2.434999942779541, 5.577000141143799, 2.5899999141693115, 2.303999900817871, 5.419000148773193, 2.500999927520752, 2.2660000324249268, 5.440999984741211, 2.696000099182129, 2.2139999866485596, 5.48199987411499, 2.693000078201294, 2.122999906539917, 5.441999912261963, 2.7869999408721924, 2.253999948501587, 5.4670000076293945, 2.678999900817871, 2.2070000171661377, 5.579999923706055, 2.6050000190734863, 2.302000045776367, 5.265999794006348, 2.6040000915527344, 2.2070000171661377, 5.234000205993652, 2.5290000438690186, 2.3519999980926514, 5.224999904632568, 2.690999984741211, 2.3459999561309814, 5.239999771118164, 2.7119998931884766, 2.5329999923706055, 5.434999942779541, 2.7009999